In [2]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

pd.set_option('display.max_columns',None)
pd.set_option('display.float_format', '{:.4f}'.format)

In [4]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

train = pd.read_csv("../../data/processed/train_fe.csv")

In [24]:
# ============================================================
# 3. 변수 제거 전략 정의
# - 6번 파일에서 만든 파생변수가 실제로 도움이 되는지 검증한다.
# - 기존 원본 변수(Name, Ticket, Cabin)는 모델에 직접 넣기 어렵기 때문에 
#   파생변수 기준으로 제거/유지 실험을 한다.
# ============================================================

feature_sets = {
    # 기본 전략 :
    # 원본 식별자/문자열 변수는 제거하고,
    # 6번 파일에서 만든 파생변수는 사용한다.
    'derived_features_only' :[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin'        
    ],

    # Title 제거:
    # Name에서 추출한 Title 파생변수가 성능에 필요한지 확인한다.
    'Without_title_info' : [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'Title'
    ],

    # TicketPrefix 제거:
    # Ticket에서 추출한 TicketPrefix 파생변수가 성능에 필요한지 확인한다.
    'Without_ticket_info':[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'TicketPrefix'
    ],

    # CabinDeck 제거:
    # Cabin에서 추출한 CabinDeck 파생변수가 성능에 필요한지 확인한다.
    'Without_CabinDeck_info' : [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'CabinDeck'
    ],

    # AgeBand, FareBand 제거:
    # 구간화한 변수보다 원본 Age, Fare, Fare_log 중심이 나은지 확인한다.
    'Without_bands':[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'AgeBand',
        'FareBand'
    ],

    # FamilySize 제거:
    # 가족 수 파생변수가 성능에 실제로 기여하는지 확인
    'Without_family_size': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'FamilySize'
    ],

    # IsAlone 제거:
    # 혼자 여부 파생변수가 성능에 기여하는지 확인
    'Without_is_alone': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'IsAlone'
    ],

    # Family 관련 변수 모두 제거:
    # FamilySize + IsAlone 동시에 제거했을 때 영향 확인
    'Without_family_info': [
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'FamilySize',
        'IsAlone'
    ],

    # 파생변수 대부분 제거:
    # 원본 기본 변수 중심의 단순 모델과 비교한다.
    'basic_features_only':[
        'PassengerId',
        'Name',
        'Ticket',
        'Cabin',
        'Title',
        'FamilySize',
        'IsAlone',
        'AgeBand',
        'Fare_log',
        'FareBand',
        'CabinDeck',
        'TicketPrefix'
    ]
}

In [25]:
# ============================================================
# 4. StratifiedKFold 설정
# - 생존/사망 비율이 fold마다 비슷하게 유지되도록 한다.
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [26]:
# ============================================================
# 5. BaseLine 모델 설정
# - 변수 세트 비교가 목적이므로 모델은 하나로 고정한다.
# ============================================================

model_params = {
    'n_estimators' : 300,
    'max_depth' : 5,
    'random_state' : 42
}

In [27]:
# ============================================================
# 6. 변수 세트별 교차검증 수행
# ============================================================

feature_set_results = []

for set_name, drop_cols in feature_sets.items():
    print("=" * 60)
    print(f"[Feature Set] {set_name}")
    print("=" * 60)

    # 현재 변수 세트 기준으로 컬럼 제거
    train_model = train.drop(columns=drop_cols)

    # 입력 변수 x와 정답 y 분리
    X = train_model.drop(columns=['Survived'])
    y = train_model['Survived']

    # 범주형 변수 원-핫 인코딩
    X = pd.get_dummies(X, drop_first=True)

    print("사용 feature 개수:", X.shape[1])

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X,y),1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        rf = RandomForestClassifier(**model_params)

        rf.fit(X_train, y_train)

        valid_pred = rf.predict(X_valid)

        acc = accuracy_score(y_valid, valid_pred)
        prec = precision_score(y_valid, valid_pred, zero_division=0)
        rec = recall_score(y_valid, valid_pred, zero_division=0)
        f1 = f1_score(y_valid, valid_pred, zero_division=0)

        feature_set_results.append({
            'feature_set' : set_name,
            'fold' : fold,
            'accuracy' : acc,
            'precision':prec,
            'recall':rec,
            'f1_score':f1,
            'n_features': X.shape[1]
        })

        print(f"""
            [Fold {fold}]
            acc={acc:.4f}
            precision={prec:.4f}
            recall={rec:.4f}
            f1={f1:.4f}
            """
        )

[Feature Set] derived_features_only
사용 feature 개수: 37

            [Fold 1]
            acc=0.8492
            precision=0.8387
            recall=0.7536
            f1=0.7939
            

            [Fold 2]
            acc=0.8258
            precision=0.7937
            recall=0.7353
            f1=0.7634
            

            [Fold 3]
            acc=0.8202
            precision=0.8333
            recall=0.6618
            f1=0.7377
            

            [Fold 4]
            acc=0.8202
            precision=0.7903
            recall=0.7206
            f1=0.7538
            

            [Fold 5]
            acc=0.8539
            precision=0.8209
            recall=0.7971
            f1=0.8088
            
[Feature Set] Without_title_info
사용 feature 개수: 33

            [Fold 1]
            acc=0.8212
            precision=0.8491
            recall=0.6522
            f1=0.7377
            

            [Fold 2]
            acc=0.8371
            precision=0.8545
           

| 변수   | 이름        | 한글  | 기준        |
| ---- | --------- | --- | --------- |
| acc  | accuracy  | 정확도 | 전체 기준 |
| prec | precision | 정밀도 | 예측 기준 |
| rec  | recall    | 재현율 | 실제 기준 |


In [28]:
# ============================================================
# 7. 변수 세트별 결과 정리
# ============================================================

feature_set_results_df = pd.DataFrame(feature_set_results)

print("\n[변수 세트별 Fold 성능]")
display(feature_set_results_df)


[변수 세트별 Fold 성능]


,feature_set,fold,accuracy,precision,recall,f1_score,n_features
0,derived_features_only,1,0.8492,0.8387,0.7536,0.7939,37
1,derived_features_only,2,0.8258,0.7937,0.7353,0.7634,37
2,derived_features_only,3,0.8202,0.8333,0.6618,0.7377,37
3,derived_features_only,4,0.8202,0.7903,0.7206,0.7538,37
4,derived_features_only,5,0.8539,0.8209,0.7971,0.8088,37
5,Without_title_info,1,0.8212,0.8491,0.6522,0.7377,33
6,Without_title_info,2,0.8371,0.8545,0.6912,0.7642,33
7,Without_title_info,3,0.7978,0.8077,0.6176,0.7000,33
8,Without_title_info,4,0.8202,0.7903,0.7206,0.7538,33
9,Without_title_info,5,0.8202,0.8246,0.6812,0.7460,33


In [29]:
# ============================================================
# 8. 변수 세트별 평균 성능 비교
# ============================================================

feature_set_summary_df = (
    feature_set_results_df
    .groupby('feature_set')
    .agg(
        accuracy_mean = ('accuracy', 'mean'),
        accuracy_std = ('accuracy', 'std'),
        precision_mean = ('precision', 'mean'),
        precision_std = ('precision', 'std'),
        recall_mean = ('recall', 'mean'),
        recall_std = ('recall', 'std'),
        f1_mean = ('f1_score', 'mean'),
        f1_std = ('f1_score', 'std'),
        n_features = ('n_features', 'mean')
    )
    .reset_index()
    .sort_values(by='f1_mean', ascending=False)
)

print("\n[변수 세트별 평균 성능 비교]")
display(feature_set_summary_df)


[변수 세트별 평균 성능 비교]


,feature_set,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,n_features
8,derived_features_only,0.8339,0.0164,0.8154,0.0223,0.7337,0.0494,0.7715,0.0292,37.0000
1,Without_bands,0.8339,0.0191,0.8153,0.0201,0.7336,0.0623,0.7711,0.0357,30.0000
5,Without_ticket_info,0.8305,0.0124,0.8178,0.0165,0.7191,0.0459,0.7644,0.0253,30.0000
0,Without_CabinDeck_info,0.8294,0.0132,0.8129,0.0158,0.7220,0.0479,0.7639,0.0267,29.0000
4,Without_is_alone,0.8283,0.0119,0.8145,0.0184,0.7162,0.0424,0.7614,0.0238,36.0000
2,Without_family_info,0.8249,0.0107,0.8068,0.0180,0.7162,0.0474,0.7578,0.0235,35.0000
3,Without_family_size,0.8238,0.0132,0.8118,0.0178,0.7044,0.0426,0.7537,0.0253,36.0000
7,basic_features_only,0.8305,0.0203,0.8700,0.0472,0.6578,0.0309,0.7486,0.0303,8.0000
6,Without_title_info,0.8193,0.0140,0.8252,0.0272,0.6725,0.0392,0.7404,0.0246,33.0000


변수 세트별 교차검증 결과, 

원본 문자열 변수는 제거하고 파생변수를 포함한 `derived_features_only` 조합이 
평균 F1 score 0.7715로 가장 높은 성능을 보였다. 

다만 `Without_bands`와의 성능 차이는 매우 미미하여, AgeBand와 FareBand는 모델 성능 개선에 크게 기여하지 않는 것으로 보인다

특히 Title과 FamilySize 제거 시 성능이 유의미하게 하락하여, 해당 파생변수들이 생존 예측에 중요한 역할을 하는 것으로 해석된다. 

반면 AgeBand/FareBand는 제거해도 성능 차이가 거의 없어 필수 변수는 아닌 것으로 판단된다.

In [33]:
# ============================================================
# 9. 최종 변수 세트 선택
# ============================================================

best_feature_set = feature_set_summary_df.iloc[0]['feature_set']
best_f1 = feature_set_summary_df.iloc[0]['f1_mean']

print("\n[최종 선택 변수 세트]")
print("Best Feature Set :", best_feature_set)
print(f"Mean F1 Score : {best_f1:.4f}")


[최종 선택 변수 세트]
Best Feature Set : derived_features_only
Mean F1 Score : 0.7715


In [34]:
# ============================================================
# 10. 결과 저장
# ============================================================

feature_set_results_df.to_csv("../../output/tables/feature_set_cv_fold_results.csv", index=False)

feature_set_summary_df.to_csv("../../output/tables/feature_set_cv_summary.csv", index=False)

print("\n변수 제거 검증 완료")


변수 제거 검증 완료
